In [1]:
import pandas as pd, xarray as xr, numpy as np, matplotlib as mpl
import matplotlib.pyplot as plt 
import pickle

In [2]:
## File path shortcuts
basepath = '/home/msaenger/Perceptions_2025/cleaned_code/'

n_members = 500

newpath = basepath + f'{n_members}_members/'

In [3]:
## Load dictionary of country-aggregated f vals for multiple k

## k = 50 to 600
with open(newpath + 'f_by_country_no_SAI.pickle', 'rb') as handle:
    f_dict = pickle.load(handle)
    
## k = 625 to 1000
with open(newpath + 'f_by_country_no_SAI_625_to_1000.pickle', 'rb') as handle:
    f_dict_v2 = pickle.load(handle)    
    
  ## k = 1050 to 1500
with open(newpath + 'f_by_country_no_SAI_1050_to_1500.pickle', 'rb') as handle:
    f_dict_v3 = pickle.load(handle)   

In [4]:
f_dict = f_dict | f_dict_v2 | f_dict_v3
f_dict

{50: <xarray.DataArray 'f' (country: 179, time: 600, ensemble: 500)> Size: 430MB
 array([[[0.2       , 0.2       , 0.2       , ..., 0.2       ,
          0.2       , 0.2       ],
         [0.29974614, 0.21297247, 0.20741369, ..., 0.20188152,
          0.30298503, 0.21014127],
         [0.38784992, 0.21669485, 0.22860573, ..., 0.21159204,
          0.29869662, 0.24808113],
         ...,
         [0.55955372, 0.41260129, 0.90438705, ..., 0.631996  ,
          0.74881812, 0.44951487],
         [0.62778614, 0.41653173, 0.87058144, ..., 0.59807406,
          0.70906233, 0.49125378],
         [0.63999841, 0.42191377, 0.86113589, ..., 0.56099003,
          0.65600486, 0.4636074 ]],
 
        [[0.2       , 0.2       , 0.2       , ..., 0.2       ,
          0.2       , 0.2       ],
         [0.29006013, 0.20532782, 0.20907546, ..., 0.20091466,
          0.20022897, 0.47400589],
         [0.2967303 , 0.20957081, 0.27051859, ..., 0.20043887,
          0.20411166, 0.65825842],
 ...
         [0.515

In [5]:
with open(newpath + 'f_by_country_no_SAI_k=50_to_1500.pickle', 'wb') as handle:
    pickle.dump(f_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [6]:
## k = 50 to 1500
with open(newpath + 'f_by_country_no_SAI_k=50_to_1500.pickle', 'rb') as handle:
    f_dict = pickle.load(handle)

f_dict

{50: <xarray.DataArray 'f' (country: 179, time: 600, ensemble: 500)> Size: 430MB
 array([[[0.2       , 0.2       , 0.2       , ..., 0.2       ,
          0.2       , 0.2       ],
         [0.29974614, 0.21297247, 0.20741369, ..., 0.20188152,
          0.30298503, 0.21014127],
         [0.38784992, 0.21669485, 0.22860573, ..., 0.21159204,
          0.29869662, 0.24808113],
         ...,
         [0.55955372, 0.41260129, 0.90438705, ..., 0.631996  ,
          0.74881812, 0.44951487],
         [0.62778614, 0.41653173, 0.87058144, ..., 0.59807406,
          0.70906233, 0.49125378],
         [0.63999841, 0.42191377, 0.86113589, ..., 0.56099003,
          0.65600486, 0.4636074 ]],
 
        [[0.2       , 0.2       , 0.2       , ..., 0.2       ,
          0.2       , 0.2       ],
         [0.29006013, 0.20532782, 0.20907546, ..., 0.20091466,
          0.20022897, 0.47400589],
         [0.2967303 , 0.20957081, 0.27051859, ..., 0.20043887,
          0.20411166, 0.65825842],
 ...
         [0.515

In [7]:
k_vals = list(f_dict.keys())

### [Across all possible k]: In each ensemble member, for each country, find the first time step where f reaches or exceeds 0.66. Save in dictionary.

In [8]:
threshold = 0.66

f_crossing_dates = {}

for k in k_vals:
    
    f_by_country = f_dict[k]

    # Order axes explicitly so time is last -> argmax works along axis=-1
    f_vals = f_by_country.transpose('country', 'ensemble', 'time').values   # (n_countries, E, T)
    time_vals = f_by_country['time'].values

    reached = f_vals >= threshold                 # bool array, same shape
    crossed = reached.any(axis=-1)                # (n_countries, E) -- did it ever cross 0.66 at all?
    first_idx = np.argmax(reached, axis=-1)        # (n_countries, E) -- index of first True

    # argmax returns 0 for rows that are all False (never crossed) -- that's misleading,
    # so explicitly mask those out to a sentinel / NaT instead of a false "time step 0"
    first_idx_masked = np.where(crossed, first_idx, -1)
    first_time = np.where(crossed, time_vals[first_idx], np.datetime64('NaT'))

    crossing_ds = xr.Dataset({
            'first_time_idx': (('country', 'ensemble'), first_idx_masked),
            'first_time': (('country', 'ensemble'), first_time)},
        coords={'country': f_by_country['country'].values,
                'ensemble': f_by_country['ensemble'].values})
    
    f_crossing_dates[k] = crossing_ds

In [9]:
## Save dict of f threshold crossings

with open(newpath + 'f_thresh_crossing_dates_v3.pickle', 'wb') as handle:
    pickle.dump(f_crossing_dates, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [10]:
## Load saved dict

with open(newpath + 'f_thresh_crossing_dates_v3.pickle', 'rb') as handle:
    f_crossing_dates2 = pickle.load(handle)
    
f_crossing_dates2 == f_crossing_dates

True

In [11]:
f_crossing_dates[1000].first_time

<xarray.DataArray 'first_time' (country: 179, ensemble: 500)> Size: 716kB
array([[cftime.DatetimeNoLeap(2061, 5, 16, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2044, 9, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2059, 6, 15, 0, 0, 0, 0, has_year_zero=True),
        ...,
        cftime.DatetimeNoLeap(2053, 6, 15, 0, 0, 0, 0, has_year_zero=True),
        None,
        cftime.DatetimeNoLeap(2018, 3, 16, 0, 0, 0, 0, has_year_zero=True)],
       [cftime.DatetimeNoLeap(2042, 8, 16, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2032, 8, 16, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2024, 4, 15, 0, 0, 0, 0, has_year_zero=True),
        ...,
        cftime.DatetimeNoLeap(2032, 6, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2024, 2, 13, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2037, 9, 15, 0, 0, 0, 0, has_year_zero=True)],
       [cftime.DatetimeNoLeap(2046, 9, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2042, 2, 13, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2039, 6, 15, 0, 0, 0, 0, has_year_zero=True),
        ...,
        cftime.DatetimeNoLeap(2039, 6, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2047, 3, 16, 0, 0, 0, 0, has_year_zero=True),
...
        cftime.DatetimeNoLeap(2040, 8, 16, 0, 0, 0, 0, has_year_zero=True),
        ...,
        cftime.DatetimeNoLeap(2052, 4, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2057, 1, 16, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2018, 4, 15, 0, 0, 0, 0, has_year_zero=True)],
       [cftime.DatetimeNoLeap(2057, 8, 16, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2045, 10, 16, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2039, 6, 15, 0, 0, 0, 0, has_year_zero=True),
        ...,
        cftime.DatetimeNoLeap(2057, 11, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2047, 8, 16, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2018, 4, 15, 0, 0, 0, 0, has_year_zero=True)],
       [cftime.DatetimeNoLeap(2057, 11, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2050, 9, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2051, 9, 15, 0, 0, 0, 0, has_year_zero=True),
        ...,
        cftime.DatetimeNoLeap(2053, 9, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2054, 9, 15, 0, 0, 0, 0, has_year_zero=True),
        cftime.DatetimeNoLeap(2045, 10, 16, 0, 0, 0, 0, has_year_zero=True)]],
      dtype=object)
Coordinates:
  * country   (country) object 1kB 'United Kingdom' ... 'European Union'
  * ensemble  (ensemble) int64 4kB 0 1 2 3 4 5 6 ... 493 494 495 496 497 498 499